In [ ]:
#| default_exp config

# Config

> Application configuration for FileSorter — path resolution, directory creation, and YAML config loading.

In [ ]:
#| export
from pydantic import BaseModel, Field
from pathlib import Path
import yaml

def _default_domain_config_dir() -> Path:
    """Resolve the bundled `config/` directory; falls back to cwd when `__file__` is unavailable (e.g. notebooks)."""
    try: return Path(__file__).parent.parent / "config"
    except NameError: return Path.cwd() / "config"

In [ ]:
#| export
class AppConfig(BaseModel):
    """Application configuration for FileSorter. Paths auto-resolve relative to `source_dir`."""
    source_dir: Path = Field(default_factory=lambda: Path.home() / "Downloads")
    sorted_dir: Path | None = None
    to_read_dir: Path | None = None
    filesystem_dir: Path | None = None
    braincave_dir: Path | None = None
    domain_config_dir: Path = Field(default_factory=_default_domain_config_dir)
    preview_lines: int = 20

    def model_post_init(self, __context):
        if self.sorted_dir is None: self.sorted_dir = self.source_dir / "sorted"
        if self.to_read_dir is None: self.to_read_dir = self.sorted_dir / "to-read"
        if self.filesystem_dir is None: self.filesystem_dir = self.sorted_dir / "filesystem"
        if self.braincave_dir is None: self.braincave_dir = self.sorted_dir / "braincave"

    def ensure_dirs(self):
        """Create sorted directory structure if it doesn't exist."""
        for d in (self.sorted_dir, self.to_read_dir, self.filesystem_dir, self.braincave_dir):
            d.mkdir(parents=True, exist_ok=True)

In [ ]:
#| export
def load_config(config_path: Path | None = None, **overrides) -> AppConfig:
    """Load config from YAML file if it exists, with optional overrides."""
    data = {}
    if config_path and config_path.exists():
        data = yaml.safe_load(config_path.read_text()) or {}
    data.update({k: v for k, v in overrides.items() if v is not None})
    return AppConfig(**data)

## Tests

In [ ]:
from fastcore.test import *
import tempfile

# Test default config
cfg = AppConfig()
test_eq(cfg.source_dir, Path.home() / "Downloads")
test_eq(cfg.sorted_dir, cfg.source_dir / "sorted")
test_eq(cfg.to_read_dir, cfg.sorted_dir / "to-read")
test_eq(cfg.filesystem_dir, cfg.sorted_dir / "filesystem")
test_eq(cfg.preview_lines, 20)

# Test custom source_dir
with tempfile.TemporaryDirectory() as tmp:
    cfg2 = AppConfig(source_dir=Path(tmp))
    test_eq(cfg2.sorted_dir, Path(tmp) / "sorted")
    cfg2.ensure_dirs()
    assert cfg2.sorted_dir.exists()
    assert cfg2.to_read_dir.exists()
    assert cfg2.filesystem_dir.exists()
    
print("\u2713 All config tests passed")

In [ ]:
# Test load_config with overrides
cfg3 = load_config(source_dir=Path("/tmp/test-downloads"))
test_eq(cfg3.source_dir, Path("/tmp/test-downloads"))
test_eq(cfg3.preview_lines, 20)

# Test load_config with YAML file
with tempfile.TemporaryDirectory() as tmp:
    yaml_path = Path(tmp) / "config.yaml"
    yaml_path.write_text("source_dir: /tmp/yaml-test\npreview_lines: 30\n")
    cfg4 = load_config(yaml_path)
    test_eq(cfg4.source_dir, Path("/tmp/yaml-test"))
    test_eq(cfg4.preview_lines, 30)

print("\u2713 All load_config tests passed")